In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
# Excel中放判断矩阵（方阵）
file_path = r"your_data.xlsx"
sheet_name = "AHP"
jm = pd.read_excel(file_path, sheet_name=sheet_name, header=None).to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {}  # AHP主要由判断矩阵决定

eigvals, eigvecs = np.linalg.eig(jm)
idx = np.argmax(eigvals.real)
w = eigvecs[:, idx].real
w = w / np.sum(w)

n = jm.shape[0]
lambda_max = eigvals[idx].real
CI = (lambda_max - n)/(n-1) if n > 1 else 0
RI_table = {1:0,2:0,3:0.58,4:0.90,5:1.12,6:1.24,7:1.32,8:1.41,9:1.45,10:1.49}
RI = RI_table.get(n, 1.49)
CR = CI/RI if RI > 0 else 0
print("w=", w, "CI=", CI, "CR=", CR)


# AHP 层次分析法

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

AHP 层次分析法 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

专家判断清晰、指标层次明确的权重确定。

## 局限性

判断矩阵存在主观性，需通过一致性检验。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。

In [ ]:
"""
AHP 层次分析法

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "AHP 层次分析法.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
INDICATOR_COLUMNS = ["指标1", "指标2", "指标3"]  # TODO: 请填写[指标名称列表]，说明：长度必须等于判断矩阵阶数。
TODO_JUDGMENT_MATRIX = [[1, 2, 4], [1/2, 1, 3], [1/4, 1/3, 1]]  # TODO: 请填写[判断矩阵]，说明：正互反矩阵，a_ij 表示 i 相对 j 的重要性。
RI_TABLE = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45}



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    A = np.array(TODO_JUDGMENT_MATRIX, dtype=float)
    eigvals, eigvecs = np.linalg.eig(A)
    max_index = np.argmax(eigvals.real)
    lambda_max = eigvals[max_index].real
    weights = eigvecs[:, max_index].real
    weights = weights / weights.sum()
    ci = (lambda_max - A.shape[0]) / (A.shape[0] - 1)
    ri = RI_TABLE.get(A.shape[0], np.nan)
    cr = ci / ri if ri else np.nan
    result = pd.DataFrame({"指标": INDICATOR_COLUMNS, "权重": weights})
    result.loc[len(result)] = ["一致性CR", cr]
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
